## **Installation of PyMOO**

In [1]:
pip install -U pymoo

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import numpy as np
# We make a dictionary from the amino acid letter
AMINO_ACIDS = list("ACDEFGHIKLMNPQRSTVWY")
print(AMINO_ACIDS)
print(type(AMINO_ACIDS))

MAP_AMINO_ACID_TO_INT = { letter: i for i, letter in enumerate(AMINO_ACIDS)}
print(MAP_AMINO_ACID_TO_INT)
print(type(MAP_AMINO_ACID_TO_INT))

['A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y']
<class 'list'>
{'A': 0, 'C': 1, 'D': 2, 'E': 3, 'F': 4, 'G': 5, 'H': 6, 'I': 7, 'K': 8, 'L': 9, 'M': 10, 'N': 11, 'P': 12, 'Q': 13, 'R': 14, 'S': 15, 'T': 16, 'V': 17, 'W': 18, 'Y': 19}
<class 'dict'>


## **Definition of the optimization problem**
We define our problem design task as a single-objective unconstrained optimization problem.
Assuming that the antimicrobial peptide (AMP) classification model take an amino acid sequence as input and outputs the probability that the sequence is an AMP, out optimization problem aims to maximize this probability. However, `pymoo` only handles minimation problems; therefore, our fitness function will be simply the probability predicted * (-1)

Solutions will be represented as sequences of integers (0 to 19), where each interger corresponds to a particular amino acid. The lenght of these sequences will be defined by a variable called `sequence_length`.  

In [7]:
from pymoo.core.problem import Problem
# This class defines an optimization problem for the pymoo library. 
# The idea is that an evolutionary algorithm generates protein sequences, 
# and the _evaluate() function assigns a fitness value to each sequence.

class ProteinDesign(Problem): # This means that ProteinDesign inherits from the Problem class provided by pymoo
    
    def __init__(self, sequence_length: int):  # The constructor receives the length of the protein sequences that will be optimized
        super().__init__(n_var=sequence_length,    # length of the sequences
                         n_obj=1,                  # The number of objectives
                         n_ieq_constr=0,           # unconstrained optimization
                         n_eq_constr=0,
                         xl=0,                     # Each variable can take values greater than or equal to 0 (0 to 19)
                         xu=len(AMINO_ACIDS),      # 19
                         vtype=np.int32)           # The decision variables are integers
        return


    def _evaluate(self, x, out, *args, **kwargs):
        # Its function is to compute the fitness values of all candidate solutions in the population
            # x : In an evolutionary algorithm, there is a population of candidate solutions.
                # Example: x =
                    #[ [0, 5, 2, 7],
                    #  [1, 1, 3, 9],
                    #  [8, 4, 6, 2] ]
            # out : Is a dictionary where pymoo expects the evaluation results to be stored
            # "F" contains the objective values (fitness values)
        (pop_size, n_var) = x.shape # In the example: pop_size = 3 n_var = 4
        if out["F"] is None: # If the fitness vector does not exist yet, it is initialized as: [-inf, -inf, -inf]
            out["F"] = np.full((pop_size), -np.inf)
        for i in range(pop_size):# Iterates through every candidate sequence in the population
            sequence = x[i, :] # Extracting One Sequence

            # Aquí llamaríamos a nuestro modelo de clasificación; por ejemplo:
            # out["F"][i] = -1.0 * model.infer(sequence), en donde vamos a tener la probabilidad de esa seq * -1
        return # no necesitamos retornar `out`; este ya fue modificado

## **Definition of the Random Sequence Generator**

`pymoo` does not provide a built-in sampling method for generating sequences of integers; its default sampling operators are mainly designed for floating-point variables. Therefore, we need to implement our own sampling strategy.

Instead of initializing the population completely at random, we use a strategy inspired by Latin Hypercube Sampling (LHS). The goal is to ensure that all 20 amino acids are represented at every sequence position across the initial population, thereby increasing diversity and improving the exploration of the search space.

In [8]:
from pymoo.core.sampling import Sampling
from pymoo.operators.sampling.rnd import IntegerRandomSampling


class DiscreteLHS(Sampling):

    # Generates the collection of integer sequences that forms the initial population.
    # problem : an instance of our `ProteinDesign` class.
    # n_samples : the number of sequences to generate.
    # args, random_state & kwargs : these arguments can be ignored.

    # The main idea is to create protein sequences represented as integer vectors and to ensure that every amino acid is well represented in the initial population 
    def _do(self, problem, n_samples, *args, random_state=None, **kwargs):
        
        assert n_samples > problem.n_var # We need to have at least as many samples as dimensions
        population = np.empty((problem.n_var, n_samples), dtype=np.int32) # Creating the Population Matrix

        # Computing How Many Times Each Amino Acid Must Appear
        num_repetitions = n_samples // len(AMINO_ACIDS)
        # The Modulo Operator % calculates the remainder of a division operation 10 % 3  = 1 
        remaining_elements = n_samples % len(AMINO_ACIDS)
        # Example:
        # len(AMINO_ACIDS) = 20
        # n_samples = 53 # They represent the amino acid that each individual will have in a specific position.

        # num_repetitions = 2
        # remaining_elements = 13

        # For example, position 0 of the population:
        # Individual 1 -> 0
        # Individual 2 -> 1
        # Individual 3 -> 2
        # ...
        # Individual 53 -> 12

    
        for i in range(problem.n_var): # i = 0 → first amino acid position, i = 1 → second amino acid position, etc

            integer_amino_acids = list(range(len(AMINO_ACIDS))) # This produces [0, 1, 2, ..., 19]

            if remaining_elements == 0: # Every amino acid appears exactly n times
                population[i] = num_repetitions * integer_amino_acids
            else: # Otherwise:
                population[i] = (
                    num_repetitions * integer_amino_acids
                    + integer_amino_acids[:remaining_elements]
                )
            # Example: 
            # num_repetitions = 2
            # remaining_elements = 5
            # [
            # 0,1,2,...,19,
            # 0,1,2,...,19,
            # 0,1,2,3,4 # This ones apear 3 times because of the remaining elements
            # ] 
            # which contains exactly 53 elements

            population[i] = np.random.permutation(population[i])  # It shuffles the elements of row i randomly
            # Example before shuffling:
            # population[i] = [0, 1, 2, 3, 4, 5]
            # So it might become: [3, 1, 5, 0, 4, 2]
        return population.T

## **Genetic Algorithm Definition**

Our genetic algorithm will use the following parameters:

| Parameter | Value |
|----------|------|
| Population size | 300 sequences |
| Parent selection | **Binary stochastic tournament**<br>(random selection probability = 0.2) |
| Crossover | Uniform (crossover probability = 1) |
| Mutation | **Random resetting**<br>(sequence mutation probability = 0.2)<br>(residue mutation probability = 0.1) |
| Replacement | **Generational** |
| Stopping condition | 150 generations |

The operators shown in **bold** are not pre-implemented in `pymoo`, so we need to implement them ourselves.

### **Definition of selection by non-deterministic binary tournament**

In [ ]:
# Our non-deterministic binary tournament selection operator
# pop : an array of objects containing the fitness values of each solution
#       in the population. For example, 'pop[0].F' contains the fitness of
#       the first individual, 'pop[1].F' of the second, and so on.
# P : a NumPy array containing the indices of the elements that will compete
#     to be selected as parents. The shape of this array is:
#     (number of parents per offspring * pop_size) x number of competitors per tournament

def non_deterministic_binary_tournament(pop, P, **kwargs):
    # pop: the current population (each individual has a fitness value F)
    # P: pairs of indices that will compete in tournaments
    # **kwargs: extra arguments (ignored here, but required by pymoo)

    global tournament_prob  # This means the function uses a variable defined outside the function
    # probability that the winner is chosen based on fitness
    # Example:
        # tournament_prob = 0.8, means 80% of the time → best fitness wins and 20% of the time → random winner

    (n_tournaments, n_competitors) = P.shape
    # n_tournaments = number of tournaments
    # n_competitors = number of participants per tournament 

    assert n_competitors == 2  # This ensures we are doing binary tournaments only

    selection = np.full(n_tournaments, -1, dtype=np.int32) # Initialize selection array, creates an array like: [-1, -1, -1, ...] This will store the winner of each tournament.

    for i in range(n_tournaments): # We process each tournament one by one
        idx_a, idx_b = P[i, :n_competitors] # First (idx_a) and second competitor (idx_b) 

        if np.random.uniform(0, 1) <= tournament_prob: # Generates a random number in [0, 1]
            # If it is ≤ tournament_prob → use fitness-based selection
            selection[i] = idx_a if pop[idx_a].F < pop[idx_b].F else idx_b
        else: # Otherwise → random selection. This adds exploration to the algorithm.
            selection[i] = np.random.choice([idx_a, idx_b]) 

    return selection  # return indices of individuals that won the tournaments

### **Definition of random resetting mutation**

This mutation consists of generating a binary mask with the same length as the amino acid sequences in our population. Then, for each position that has a value of 1 in the binary mask, we replace the amino acid at that same position in the sequence with a different amino acid.

In [ ]:
from pymoo.core.mutation import Mutation # This imports the base class for mutation operators in pymoo
 

class RandomResetting(Mutation):

    def _do(self, problem, X, *args, random_state=None, **kwargs):
        # problem: the optimization problem
        # X: the population matrix
        # shape: (n_individuals, sequence_length)
        # args, kwargs: extra optional parameters

        # decide which residues of which sequences will be mutated
        # This defines the probability that a single gene will mutate
        # self.prob → probability of mutating an individual
        # “Do we mutate this sequence at all?”
        # Example: self.prob = 0.2 so 20% of individuals are selected for mutation

        # self.prob_var → probability per gene (per position)
        # “If a sequence is selected, which positions mutate?” 
        # Example: self.prob_var = 0.1 so each amino acid position has a 10% chance of mutating

        # Instead of doing it in two steps, the code collapses both into one:
        threshold = self.prob.value * self.prob_var.value 

        mask = np.random.rand(*X.shape) < threshold # This generates a random matrix of the same shape as X
        # Each entry is: True → mutate this amino acid, False → keep it
        indices = np.where(mask)  # It converts the matrix into coordinates of True values
        # Example: 
        # from this: [
        # [False, True ], 
        # [True,  False]
        # ] np.where(mask) returns: (array([0, 1]), array([1, 0]))...


        for i, j in zip(*indices): # Loop over selected mutations; For each selected amino acid, apply mutation
            mutant = original = X[i, j] # This saves the current amino acid before changing it.
            # Ensure mutation changes the value
            while mutant == original:
                mutant = np.random.choice(np.arange(len(AMINO_ACIDS)))
                # So if: original = 7 it keeps sampling until: mutant != 7
            X[i, j] = mutant # The selected position is replaced with the new amino acid

        return X

### **Definition of generational replacement**

In [ ]:
from pymoo.core.survival import Survival # This imports the base class that defines how individuals survive to the next generation in a genetic algorithm.


class GenerationalReplacement(Survival):
    # This create a custom survival strategy. It tells pymoo “This is how I decide who stays in the next generation.”

    def _do(self, problem, pop, *args, n_survive=None, **kwargs):
        # problem: the optimization problem
        # pop: the combined population (parents + offspring)
        # n_survive: how many individuals must survive to the next generation
        # *args, **kwargs: extra optional parameters (ignored)
        return pop[-n_survive:]